# Chunking Practical: Most Useful Methods
This notebook covers the following methods:

1. `RecursiveCharacterTextSplitter`
2. `TokenTextSplitter`
3. `CharacterTextSplitter`


4. Fixed-size chunking
5. Content-aware chunking
6. Document structure-based chunking
7. Semantic chunking

## Learning objectives

- Explain chunk size and chunk overlap
- Compare fixed, character, recursive, token, structural, and semantic splitting
- Understand why different strategies produce different boundaries
- Choose a suitable starting method for a given document type

1. Fixed-size character chunking
   → CharacterTextSplitter

2. Fixed-size token chunking
   → TokenTextSplitter

3. Recursive structure-aware chunking
   → RecursiveCharacterTextSplitter

4. Document structure-based chunking
   → MarkdownHeaderTextSplitter, HTML splitters, etc.

5. Semantic chunking
   → Embedding-based topic boundaries

CharacterTextSplitter

→ Primarily separator-based

→ Length measured using characters

TokenTextSplitter

→ Directly splits according to tokens

RecursiveCharacterTextSplitter

→ Tries to preserve paragraphs, lines and words

→ Not purely blind fixed-size chunking

In [1]:
from __future__ import annotations
import math
import re
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from langchain_text_splitters import (
    CharacterTextSplitter,
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

d:\MAHA\AIPro\AgenticAI\myAgenticAI_9PM_June26\myenv9pm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Sample Document

The sample document intentionally contains multiple topics and Markdown headings.  
This makes it useful for comparing basic, structure-aware, and semantic chunking.

In [2]:
document = """
# Artificial Intelligence

Artificial intelligence enables machines to perform tasks that normally require human intelligence.
AI systems can analyze information, identify patterns, and support decision-making.

## Machine Learning

Machine learning is a branch of artificial intelligence.
It enables systems to learn patterns from historical data instead of relying only on explicitly programmed rules.
Supervised learning uses labelled examples, while unsupervised learning discovers patterns in unlabelled data.

## Deep Learning

Deep learning uses neural networks containing multiple layers.
It is commonly applied to image recognition, speech processing, and natural-language applications.

# Employee Leave Policy

Employees receive twenty paid leave days during a calendar year.
Leave requests should be submitted to the reporting manager before the planned absence.
Emergency leave may be submitted later when advance notice is not possible.

## Approval Process

The manager reviews the leave balance, project schedule, and team availability.
Approved leave is recorded in the human-resource management system.

# Information Security

Employees must not share passwords or authentication codes.
Sensitive company information should be stored only in approved systems.
Suspected security incidents must be reported to the security team immediately.
""".strip()

In [3]:
print(document)

# Artificial Intelligence

Artificial intelligence enables machines to perform tasks that normally require human intelligence.
AI systems can analyze information, identify patterns, and support decision-making.

## Machine Learning

Machine learning is a branch of artificial intelligence.
It enables systems to learn patterns from historical data instead of relying only on explicitly programmed rules.
Supervised learning uses labelled examples, while unsupervised learning discovers patterns in unlabelled data.

## Deep Learning

Deep learning uses neural networks containing multiple layers.
It is commonly applied to image recognition, speech processing, and natural-language applications.

# Employee Leave Policy

Employees receive twenty paid leave days during a calendar year.
Leave requests should be submitted to the reporting manager before the planned absence.
Emergency leave may be submitted later when advance notice is not possible.

## Approval Process

The manager reviews the lea

In [4]:
from pypdf import PdfReader
reader = PdfReader("D:\MAHA\AIPro\AgenticAI\myAgenticAI_9PM_June26\Data\DeepSeek-V4.pdf")
page_texts = []
for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if text.strip():
        page_texts.append(text)

    print(
        f"Page {page_number}: "
        f"{len(text)} characters extracted"
    )

Page 1: 2610 characters extracted
Page 2: 2538 characters extracted
Page 3: 1881 characters extracted
Page 4: 4113 characters extracted
Page 5: 4075 characters extracted
Page 6: 1924 characters extracted
Page 7: 3482 characters extracted
Page 8: 3097 characters extracted
Page 9: 2420 characters extracted
Page 10: 2420 characters extracted
Page 11: 2331 characters extracted
Page 12: 2460 characters extracted
Page 13: 3603 characters extracted
Page 14: 2716 characters extracted
Page 15: 2860 characters extracted
Page 16: 3033 characters extracted
Page 17: 3899 characters extracted
Page 18: 3659 characters extracted
Page 19: 3456 characters extracted
Page 20: 3663 characters extracted
Page 21: 3405 characters extracted
Page 22: 3255 characters extracted
Page 23: 3503 characters extracted
Page 24: 3408 characters extracted
Page 25: 4089 characters extracted
Page 26: 3630 characters extracted
Page 27: 2873 characters extracted
Page 28: 2310 characters extracted
Page 29: 2478 characters extr

In [5]:
document = "\n\n".join(page_texts)

In [6]:
print(document)

DeepSeek-V4:
Towards Highly Efficient Million-Token Context Intelligence
DeepSeek-AI
research@deepseek.com
Abstract
We present a preview version of DeepSeek-V4 series, including two strong Mixture-of-
Experts (MoE) language models — DeepSeek-V4-Pro with 1.6T parameters (49B activated) and
DeepSeek-V4-Flash with 284B parameters (13B activated) — both supporting a context length of
one million tokens. DeepSeek-V4 series incorporate several key upgrades in architecture and op-
timization: (1) a hybrid attention architecture that combines Compressed Sparse Attention (CSA)
and Heavily Compressed Attention (HCA) to improve long-context efficiency; (2) Manifold-
Constrained Hyper-Connections (mHC) that enhance conventional residual connections; (3)
and the Muon optimizer for faster convergence and greater training stability. We pre-train
both models on more than 32T diverse and high-quality tokens, followed by a comprehensive
post-training pipeline that unlocks and further enhances their capa

In [7]:
# 2. Helper Functions

# These helpers print chunks consistently and collect simple statistics.

from typing import Any

def print_chunks(
    title: str,
    chunks: list[Any],
    preview_limit: int = 500,
) -> None:
    print(f"\n{'=' * 90}")
    print(title)
    print(f"Total chunks: {len(chunks)}")
    print("=" * 90)

    for index, chunk in enumerate(chunks, start=1):

        # Get text from a LangChain Document or normal Python object
        text = getattr(
            chunk,
            "page_content",
            str(chunk),
        )

        # Get metadata if available
        metadata = getattr(
            chunk,
            "metadata",
            {},
        )

        print(
            f"\nChunk {index} | "
            f"characters={len(text)}"
        )

        if metadata:
            print("Metadata:", metadata)

        if len(text) > preview_limit:
            print(text[:preview_limit] + "...")
        else:
            print(text)

In [8]:
from typing import Any
import numpy as np


def chunk_stats(
    method: str,
    chunks: list[Any],
) -> dict[str, Any]:

    lengths = [
        len(
            getattr(
                chunk,
                "page_content",
                str(chunk),
            )
        )
        for chunk in chunks
    ]

    return {
        "method": method,
        "number_of_chunks": len(chunks),
        "minimum_length": min(lengths) if lengths else 0,
        "maximum_length": max(lengths) if lengths else 0,
        "average_length": (
            round(float(np.mean(lengths)), 2)
            if lengths
            else 0
        ),
        "total_stored_characters": sum(lengths),
    }

# 3. Fixed-Size Chunking

Fixed-size chunking divides content after a specified number of characters or tokens.

It is simple and predictable, but it may break:

- Words
- Sentences
- Paragraphs

In [ ]:
def fixed_size_chunking(text: str, chunk_size: int, chunk_overlap: int = 0) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if chunk_overlap < 0 or chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must satisfy 0 <= chunk_overlap < chunk_size."
        )

    step = chunk_size - chunk_overlap

    return [
        text[start:start + chunk_size]
        for start in range(0, len(text), step)
    ]

In [10]:
text = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

In [11]:
len(text)

26

In [13]:
fixed_size_chunking(text, chunk_size=10, chunk_overlap=2)

['ABCDEFGHIJ', 'IJKLMNOPQR', 'QRSTUVWXYZ', 'YZ']

10 - 2 = 8

So chunks begin at positions 0, 8, 16, 24.

In [14]:
document

'DeepSeek-V4:\nTowards Highly Efficient Million-Token Context Intelligence\nDeepSeek-AI\nresearch@deepseek.com\nAbstract\nWe present a preview version of DeepSeek-V4 series, including two strong Mixture-of-\nExperts (MoE) language models — DeepSeek-V4-Pro with 1.6T parameters (49B activated) and\nDeepSeek-V4-Flash with 284B parameters (13B activated) — both supporting a context length of\none million tokens. DeepSeek-V4 series incorporate several key upgrades in architecture and op-\ntimization: (1) a hybrid attention architecture that combines Compressed Sparse Attention (CSA)\nand Heavily Compressed Attention (HCA) to improve long-context efficiency; (2) Manifold-\nConstrained Hyper-Connections (mHC) that enhance conventional residual connections; (3)\nand the Muon optimizer for faster convergence and greater training stability. We pre-train\nboth models on more than 32T diverse and high-quality tokens, followed by a comprehensive\npost-training pipeline that unlocks and further enha

In [16]:
fixed_chunks=fixed_size_chunking(text=document, chunk_size=300, chunk_overlap=50)

In [17]:
fixed_chunks

['DeepSeek-V4:\nTowards Highly Efficient Million-Token Context Intelligence\nDeepSeek-AI\nresearch@deepseek.com\nAbstract\nWe present a preview version of DeepSeek-V4 series, including two strong Mixture-of-\nExperts (MoE) language models — DeepSeek-V4-Pro with 1.6T parameters (49B activated) and\nDeepSeek-V',
 'ith 1.6T parameters (49B activated) and\nDeepSeek-V4-Flash with 284B parameters (13B activated) — both supporting a context length of\none million tokens. DeepSeek-V4 series incorporate several key upgrades in architecture and op-\ntimization: (1) a hybrid attention architecture that combines Compresse',
 'rid attention architecture that combines Compressed Sparse Attention (CSA)\nand Heavily Compressed Attention (HCA) to improve long-context efficiency; (2) Manifold-\nConstrained Hyper-Connections (mHC) that enhance conventional residual connections; (3)\nand the Muon optimizer for faster convergence an',
 ')\nand the Muon optimizer for faster convergence and greater training 

In [18]:
len(fixed_chunks)

692

In [19]:
print_chunks("Manual Fixed-Size Chunking", fixed_chunks)


Manual Fixed-Size Chunking
Total chunks: 692

Chunk 1 | characters=300
DeepSeek-V4:
Towards Highly Efficient Million-Token Context Intelligence
DeepSeek-AI
research@deepseek.com
Abstract
We present a preview version of DeepSeek-V4 series, including two strong Mixture-of-
Experts (MoE) language models — DeepSeek-V4-Pro with 1.6T parameters (49B activated) and
DeepSeek-V

Chunk 2 | characters=300
ith 1.6T parameters (49B activated) and
DeepSeek-V4-Flash with 284B parameters (13B activated) — both supporting a context length of
one million tokens. DeepSeek-V4 series incorporate several key upgrades in architecture and op-
timization: (1) a hybrid attention architecture that combines Compresse

Chunk 3 | characters=300
rid attention architecture that combines Compressed Sparse Attention (CSA)
and Heavily Compressed Attention (HCA) to improve long-context efficiency; (2) Manifold-
Constrained Hyper-Connections (mHC) that enhance conventional residual connections; (3)
and the Muon optimizer

## What to observe

- Every chunk targets the same maximum character size.
- Fifty characters are repeated between neighbouring chunks.
- A boundary may occur in the middle of a sentence or heading.
- Overlap protects some boundary information but also creates duplication.

# 4. `CharacterTextSplitter`

`CharacterTextSplitter` splits using one selected separator.

In this example, the separator is a blank line (`"\n\n"`), so it tries to keep paragraphs together.

This is easy to understand, but one separator may not be sufficient for every document.

In [20]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

In [22]:
character_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
)

In [23]:
character_chunks = character_splitter.split_text(document)

Created a chunk of size 2610, which is longer than the specified 300
Created a chunk of size 2538, which is longer than the specified 300
Created a chunk of size 1881, which is longer than the specified 300
Created a chunk of size 4113, which is longer than the specified 300
Created a chunk of size 4075, which is longer than the specified 300
Created a chunk of size 1924, which is longer than the specified 300
Created a chunk of size 3482, which is longer than the specified 300
Created a chunk of size 3097, which is longer than the specified 300
Created a chunk of size 2420, which is longer than the specified 300
Created a chunk of size 2420, which is longer than the specified 300
Created a chunk of size 2331, which is longer than the specified 300
Created a chunk of size 2460, which is longer than the specified 300
Created a chunk of size 3603, which is longer than the specified 300
Created a chunk of size 2716, which is longer than the specified 300
Created a chunk of size 2860, whic

In [25]:
print_chunks("CharacterTextSplitter", character_chunks)


CharacterTextSplitter
Total chunks: 58

Chunk 1 | characters=2610
DeepSeek-V4:
Towards Highly Efficient Million-Token Context Intelligence
DeepSeek-AI
research@deepseek.com
Abstract
We present a preview version of DeepSeek-V4 series, including two strong Mixture-of-
Experts (MoE) language models — DeepSeek-V4-Pro with 1.6T parameters (49B activated) and
DeepSeek-V4-Flash with 284B parameters (13B activated) — both supporting a context length of
one million tokens. DeepSeek-V4 series incorporate several key upgrades in architecture and op-
timization: (1) a hyb...

Chunk 2 | characters=2538
Contents
1 Introduction 4
2 Architecture 6
2.1 Designs Inherited from DeepSeek-V3 . . . . . . . . . . . . . . . . . . . . . . . . . . 7
2.2 Manifold-Constrained Hyper-Connections . . . . . . . . . . . . . . . . . . . . . . 7
2.3 Hybrid Attention with CSA and HCA . . . . . . . . . . . . . . . . . . . . . . . . . 9
2.3.1 Compressed Sparse Attention . . . . . . . . . . . . . . . . . . . . . . . . . . 9

## Important observation

`CharacterTextSplitter` is separator-driven. If an individual separator-delimited block is already larger than the target size, the output may not behave like strict fixed-width slicing.

Use it when the source has a predictable separator and you want a simple baseline.

# 5. `RecursiveCharacterTextSplitter`

This splitter tries separators recursively.

Typical order:

```text
Paragraph → line → sentence-like boundary → word → character
```

It first tries to preserve large natural units. If a unit is too large, it moves to a finer separator.

In [36]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    add_start_index=True,
)

In [40]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=30,
    chunk_overlap=2,
    length_function=len,
    add_start_index=True,
)

In [32]:
text = """Artificial Intelligence

AI enables machines to perform intelligent tasks.
Machine learning is a part of AI.

Employee Leave Policy

Employees receive 20 paid leave days.
Leave must be approved by the manager.
"""


In [37]:
recursive_splitter.create_documents(
    texts=[text],
    metadatas=[{"source": "classroom_sample.md"}],

)

[Document(metadata={'source': 'classroom_sample.md', 'start_index': 0}, page_content='Artificial Intelligence\n\nAI enables machines to perform intelligent tasks.\nMachine learning is a part of AI.\n\nEmployee Leave Policy\n\nEmployees receive 20 paid leave days.\nLeave must be approved by the manager.')]

In [38]:
recursive_documents = recursive_splitter.create_documents(
    texts=[document],
    metadatas=[{"source": "classroom_sample.md"}],
)

In [39]:
print_chunks(
    "RecursiveCharacterTextSplitter",
    recursive_documents,
)


RecursiveCharacterTextSplitter
Total chunks: 687

Chunk 1 | characters=289
Metadata: {'source': 'classroom_sample.md', 'start_index': 0}
DeepSeek-V4:
Towards Highly Efficient Million-Token Context Intelligence
DeepSeek-AI
research@deepseek.com
Abstract
We present a preview version of DeepSeek-V4 series, including two strong Mixture-of-
Experts (MoE) language models — DeepSeek-V4-Pro with 1.6T parameters (49B activated) and

Chunk 2 | characters=284
Metadata: {'source': 'classroom_sample.md', 'start_index': 290}
DeepSeek-V4-Flash with 284B parameters (13B activated) — both supporting a context length of
one million tokens. DeepSeek-V4 series incorporate several key upgrades in architecture and op-
timization: (1) a hybrid attention architecture that combines Compressed Sparse Attention (CSA)

Chunk 3 | characters=267
Metadata: {'source': 'classroom_sample.md', 'start_index': 575}
and Heavily Compressed Attention (HCA) to improve long-context efficiency; (2) Manifold-
Constrained Hyper-

## Why it is commonly used

- It works well for general text
- It tries to preserve natural boundaries
- It still controls chunk size
- It is a strong baseline for PDFs, reports, policies, and articles

# 6. `TokenTextSplitter`

LLMs and embedding models process tokens rather than raw characters.

Token-based chunking is useful when:

- The embedding model has a strict token limit
- You need predictable model-input sizing
- Character counts do not accurately represent model usage

This example uses the `cl100k_base` tokenizer.

In [48]:
token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=80,
    chunk_overlap=15,
)

In [49]:
token_chunks = token_splitter.split_text(document)


In [46]:
print_chunks("TokenTextSplitter", token_chunks)


TokenTextSplitter
Total chunks: 790

Chunk 1 | characters=313
DeepSeek-V4:
Towards Highly Efficient Million-Token Context Intelligence
DeepSeek-AI
research@deepseek.com
Abstract
We present a preview version of DeepSeek-V4 series, including two strong Mixture-of-
Experts (MoE) language models — DeepSeek-V4-Pro with 1.6T parameters (49B activated) and
DeepSeek-V4-Flash with 

Chunk 2 | characters=344
 (49B activated) and
DeepSeek-V4-Flash with 284B parameters (13B activated) — both supporting a context length of
one million tokens. DeepSeek-V4 series incorporate several key upgrades in architecture and op-
timization: (1) a hybrid attention architecture that combines Compressed Sparse Attention (CSA)
and Heavily Compressed Attention (HCA)

Chunk 3 | characters=350
 (CSA)
and Heavily Compressed Attention (HCA) to improve long-context efficiency; (2) Manifold-
Constrained Hyper-Connections (mHC) that enhance conventional residual connections; (3)
and the Muon optimizer for faster converge